<a href="https://colab.research.google.com/github/zombimann/Mathematical-video-animations-and-visualization/blob/main/_kids_law_of_large_numbers_rolling_balls.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Educational Animation: The Law of Large Numbers

## Overview for Educators and Parents
This notebook contains a complete Python implementation for generating an educational animation designed to teach middle school students about the Law of Large Numbers (LLN). The animation uses a physical metaphor—rolling balls down a split ramp—to visualize how random individual events aggregate into predictable mathematical patterns.

## Learning Objectives
1. Understand that individual random events (a single ball roll) are unpredictable.
2. Observe how the cumulative frequency of an event (proportion of balls going left) fluctuates significantly when the sample size is small.
3. Visualize the convergence of these proportions toward the theoretical probability (0.5 or 50%) as the number of trials increases.

## Visual Components
* **Ramp Simulation**: A visual representation of balls falling through a 50/50 splitter into two bins (Left and Right). This provides a physical intuition for probability.
* **Convergence Chart**: A logarithmic line graph that tracks the 'running proportion' of balls that went left. This demonstrates the mathematical 'settling' of data.
* **Observation Panel**: Real-time bullet points that guide the viewer to notice key phases of the experiment, from early volatility to eventual stability.

## Technical Specifications
* **Output Format**: Vertical 1080x1920 (9:16 aspect ratio), optimized for mobile devices and social learning platforms like YouTube Shorts.
* **Libraries Used**: Matplotlib (for rendering and animation), NumPy (for statistical simulation), and FFmpeg (for video encoding).

In [3]:
"""
Author: Mugambi Ndwiga
Instagram: @craftsandengineering
GitHub: github.com/zombimann/Mathematical-video-animations-and-visualization
Level: Middle School
Concept: The Law of Large Numbers, shown by rolling balls down a split (50/50) ramp.
Format: vertical 1080x1920 (9:16) for mobile viewing / YouTube Shorts.

One ball is unpredictable. As more balls roll, the fraction going left stops
swinging and settles onto the true probability p = 1/2. That settling is the
Law of Large Numbers:  p_hat_n = L_n / n  ->  p = 1/2  as  n -> infinity.
"""

import numpy as np
import matplotlib
matplotlib.use("Agg")  # render straight to file; no stray inline figure
import matplotlib.pyplot as plt
from matplotlib.patches import Polygon, Rectangle, FancyBboxPatch
from matplotlib.animation import FuncAnimation, FFMpegWriter

try:
    import imageio_ffmpeg
    plt.rcParams["animation.ffmpeg_path"] = imageio_ffmpeg.get_ffmpeg_exe()
except Exception:
    pass

# --------------------------------------------------------------------------- #
# CONFIG  — everything tweakable lives here (parametric by design)
# --------------------------------------------------------------------------- #
CONFIG = dict(
    seed=7, p_left=0.5, n_total=400, n_individual=12,
    fps=24, frames_intro=54, frames_per_ball=16, frames_batch=486,
    frames_reveal=96, frames_close=42, n_stream=4, stream_speed=0.05,
    fig_w=10.8, fig_h=19.2, dpi=100,                 # -> 1080 x 1920 (9:16)
    outfile="law_of_large_numbers_vertical.mp4",
)
COLORS = dict(bg="#FFFFFF", grid="#E0E0E0", ink="#2A3D66",
              sage="#86A788", coral="#E5989B", mustard="#E9C46A", navy="#2A3D66",
              grey="#6B6B6B", target="#9AA0A6")
plt.rcParams.update({"font.family": "sans-serif",
                     "font.sans-serif": ["Inter", "Roboto", "DejaVu Sans"],
                     "text.color": COLORS["ink"]})
C = CONFIG

# --------------------------------------------------------------------------- #
# THE EXPERIMENT  — precompute outcomes and the running proportion p_hat_n
# --------------------------------------------------------------------------- #
rng = np.random.default_rng(C["seed"])
goes_left = rng.random(C["n_total"]) < C["p_left"]            # True = LEFT
lefts_cum = np.concatenate(([0], np.cumsum(goes_left.astype(int))))
trials = np.arange(C["n_total"] + 1)
proportion = np.divide(lefts_cum, trials,
                       out=np.full_like(trials, np.nan, float), where=trials > 0)

# --------------------------------------------------------------------------- #
# RAMP GEOMETRY (data coords; ranges matched to the axes box -> no distortion)
# --------------------------------------------------------------------------- #
DROP_TOP = 8.8
APEX = (0.0, 5.8)
LEFT_X, RIGHT_X = -4.2, 4.2
BIN_BOTTOM, BIN_H, BAR_W = 0.2, 4.3, 2.4
FILL_CAP = C["n_total"] * 0.62
RAMP_XLIM = (-9.1, 9.1)
RAMP_YLIM = (-1.4, 11.0)

def smoother(t):
    t = np.clip(t, 0.0, 1.0)
    return t * t * t * (t * (6 * t - 15) + 10)

def bar_height(count):
    # sqrt scaling: small early counts stay visible; equal counts -> equal heights
    return BIN_H * min((count / FILL_CAP) ** 0.5, 1.0)

def ball_position(side_left, t):
    """Vertical drop onto the apex, then a smooth arc into a bin."""
    t = float(np.clip(t, 0.0, 1.0))
    if t <= 0.4:
        u = smoother(t / 0.4)
        return 0.0, DROP_TOP + (APEX[1] - DROP_TOP) * u
    u = smoother((t - 0.4) / 0.6)
    bx = LEFT_X if side_left else RIGHT_X
    p0, p1, p2 = np.array(APEX), np.array([0.6 * bx, 5.3]), np.array([bx, 1.6])
    p = (1 - u) ** 2 * p0 + 2 * (1 - u) * u * p1 + u ** 2 * p2
    return float(p[0]), float(p[1])

# --------------------------------------------------------------------------- #
# TIMELINE
# --------------------------------------------------------------------------- #
INTRO_END = C["frames_intro"]
IND_END = INTRO_END + C["n_individual"] * C["frames_per_ball"]
BATCH_END = IND_END + C["frames_batch"]
REVEAL_END = BATCH_END + C["frames_reveal"]
CLOSE_START = REVEAL_END
TOTAL = REVEAL_END + C["frames_close"]

def frame_state(f):
    if f < INTRO_END:
        return dict(n=0, phase="intro", flights=[])
    if f < IND_END:
        fl = f - INTRO_END
        idx = fl // C["frames_per_ball"]
        sub = (fl % C["frames_per_ball"]) / C["frames_per_ball"]
        return dict(n=int(idx), phase="few", flights=[(bool(goes_left[idx]), sub, 1.0)])
    if f < BATCH_END:
        fl = f - IND_END
        frac = fl / C["frames_batch"]
        n = min(C["n_individual"] + int(round((C["n_total"] - C["n_individual"]) * frac ** 2)),
                C["n_total"])
        flights = []
        for j in range(C["n_stream"]):
            t = (fl * C["stream_speed"] + j / C["n_stream"]) % 1.0
            side = ((j + fl // 7) % 2 == 0)
            flights.append((side, t, 0.5))
        return dict(n=int(n), phase="many", flights=flights)
    if f < REVEAL_END:
        return dict(n=C["n_total"], phase="reveal", flights=[])
    return dict(n=C["n_total"], phase="close", flights=[])

def fade(f, start, dur=16):
    return float(np.clip((f - start) / dur, 0.0, 1.0))

# --------------------------------------------------------------------------- #
# FIGURE & STATIC SCENERY  — vertical stack: title / ramp / chart / card
# --------------------------------------------------------------------------- #
fig = plt.figure(figsize=(C["fig_w"], C["fig_h"]), dpi=C["dpi"])
fig.patch.set_facecolor(COLORS["bg"])
ax_ramp = fig.add_axes([0.05, 0.560, 0.90, 0.345]); ax_ramp.set_facecolor(COLORS["bg"])
ax_conv = fig.add_axes([0.135, 0.310, 0.83, 0.200]); ax_conv.set_facecolor(COLORS["bg"])
ax_panel = fig.add_axes([0.05, 0.045, 0.90, 0.220]); ax_panel.set_facecolor(COLORS["bg"])
ax_close = fig.add_axes([0, 0, 1, 1], zorder=20); ax_close.set_xlim(0, 1); ax_close.set_ylim(0, 1)
ax_close.axis("off"); ax_close.set_visible(False)

# ---- Ramp panel ----
ax_ramp.set_xlim(*RAMP_XLIM); ax_ramp.set_ylim(*RAMP_YLIM)
ax_ramp.set_xticks(range(-8, 9, 2)); ax_ramp.set_yticks(range(0, 11, 2))
ax_ramp.tick_params(labelbottom=False, labelleft=False, length=0)
ax_ramp.grid(True, color=COLORS["grid"], lw=1.0)
for s in ax_ramp.spines.values():
    s.set_color(COLORS["grid"])
ax_ramp.add_patch(Polygon([APEX, (-1.3, 4.5), (1.3, 4.5)], closed=True,
                          fc=COLORS["navy"], ec=COLORS["ink"], lw=2, zorder=4))
ax_ramp.plot([0, 0], [APEX[1], DROP_TOP], color=COLORS["grey"], lw=1.6, ls=(0, (4, 4)), zorder=2)
for cx in (LEFT_X, RIGHT_X):
    ax_ramp.add_patch(Rectangle((cx - 1.4, BIN_BOTTOM), 2.8, BIN_H,
                                fill=False, ec=COLORS["ink"], lw=2, zorder=3))
bar_left = Rectangle((LEFT_X - BAR_W / 2, BIN_BOTTOM), BAR_W, 0.0,
                     fc=COLORS["sage"], ec=COLORS["ink"], lw=1.5, zorder=3)
bar_right = Rectangle((RIGHT_X - BAR_W / 2, BIN_BOTTOM), BAR_W, 0.0,
                      fc=COLORS["coral"], ec=COLORS["ink"], lw=1.5, zorder=3)
ax_ramp.add_patch(bar_left); ax_ramp.add_patch(bar_right)
ax_ramp.text(LEFT_X, -0.85, "LEFT", ha="center", va="top", fontsize=18,
             fontweight="bold", color=COLORS["sage"])
ax_ramp.text(RIGHT_X, -0.85, "RIGHT", ha="center", va="top", fontsize=18,
             fontweight="bold", color=COLORS["coral"])
count_left = ax_ramp.text(LEFT_X, 4.7, "", ha="center", fontsize=17, color=COLORS["ink"])
count_right = ax_ramp.text(RIGHT_X, 4.7, "", ha="center", fontsize=17, color=COLORS["ink"])
apex_label = ax_ramp.text(0, 9.9, "$p=\\dfrac{1}{2}$", ha="center", va="center", fontsize=22,
                          color=COLORS["ink"], alpha=0,
                          bbox=dict(boxstyle="round,pad=0.3", fc="white", ec=COLORS["grid"]))
arrow_kw = dict(arrowstyle="-|>", color=COLORS["ink"], lw=1.6)
arr_l = ax_ramp.annotate("", xy=(-2.2, 4.9), xytext=(-0.35, 5.7), arrowprops=arrow_kw, alpha=0)
arr_r = ax_ramp.annotate("", xy=(2.2, 4.9), xytext=(0.35, 5.7), arrowprops=arrow_kw, alpha=0)
counter_n = ax_ramp.text(-8.7, 10.3, "", ha="left", fontsize=23, fontweight="bold",
                         color=COLORS["ink"])
counter_phat = ax_ramp.text(-8.7, 9.2, "", ha="left", fontsize=19, color=COLORS["navy"])
flight_dots = [ax_ramp.plot([], [], "o", ms=17, mec=COLORS["ink"], mew=1.5, zorder=6)[0]
               for _ in range(max(C["n_stream"], 1))]

# ---- Convergence chart (wide + short) ----
ax_conv.set_xscale("log"); ax_conv.set_xlim(1, C["n_total"]); ax_conv.set_ylim(0, 1)
ax_conv.set_xticks([1, 10, 100, C["n_total"]]); ax_conv.set_xticklabels([1, 10, 100, C["n_total"]])
ax_conv.set_yticks([0, 0.5, 1.0])
ax_conv.tick_params(labelsize=14, colors=COLORS["grey"])
for s in ("top", "right"):
    ax_conv.spines[s].set_visible(False)
for s in ("left", "bottom"):
    ax_conv.spines[s].set_color(COLORS["grey"])
ax_conv.grid(True, color=COLORS["grid"], lw=0.8)
ax_conv.axhline(C["p_left"], color=COLORS["target"], lw=1.8, ls=(0, (5, 4)))
ax_conv.text(C["n_total"], C["p_left"] + 0.05, "true  $p=1/2$", ha="right", fontsize=14,
             color=COLORS["grey"])
ax_conv.set_xlabel("number of balls,  $n$", fontsize=16, color=COLORS["ink"])
ax_conv.set_ylabel("fraction left,  $\\hat{p}_n$", fontsize=16, color=COLORS["ink"])
ax_conv.set_title("Running proportion settles down", fontsize=17, fontweight="bold",
                  color=COLORS["ink"], pad=8)
ax_conv.text(0.97, 0.12, "$\\hat{p}_n = L_n / n$", transform=ax_conv.transAxes,
             ha="right", fontsize=16, color=COLORS["navy"],
             bbox=dict(boxstyle="round,pad=0.3", fc="white", ec=COLORS["grid"]))
conv_note = ax_conv.text(0.32, 0.70, "settles near 1/2", transform=ax_conv.transAxes,
                         fontsize=15, color=COLORS["ink"], alpha=0, fontweight="bold")
(conv_line,) = ax_conv.plot([], [], color=COLORS["navy"], lw=2.6, zorder=4)
(conv_head,) = ax_conv.plot([], [], "o", ms=11, color=COLORS["navy"], mec="white", mew=1.5, zorder=5)

# ---- Annotation card (full width, single column) ----
ax_panel.set_xlim(0, 1); ax_panel.set_ylim(0, 1); ax_panel.axis("off")
ax_panel.add_patch(FancyBboxPatch((0.02, 0.03), 0.96, 0.94,
                                  boxstyle="round,pad=0.02,rounding_size=0.03",
                                  fc="#FBFBFB", ec=COLORS["grid"], lw=1.5,
                                  transform=ax_panel.transAxes))
ax_panel.text(0.05, 0.90, "What to notice", fontsize=18, fontweight="bold", color=COLORS["ink"])
bullets = [(COLORS["sage"], "Each roll: left or right, 50/50"),
           (COLORS["coral"], "A few rolls swing wildly"),
           (COLORS["mustard"], "Many rolls settle near 1/2"),
           (COLORS["navy"], "Law of Large Numbers:  $\\hat{p}_n \\to \\frac{1}{2}$")]
bullet_dots, bullet_txts = [], []
ys = [0.69, 0.50, 0.31, 0.12]
for (col, txt), y in zip(bullets, ys):
    d = ax_panel.plot(0.06, y + 0.01, "o", ms=14, color=col, mec=COLORS["ink"], mew=1.0, alpha=0)[0]
    t = ax_panel.text(0.11, y, txt, fontsize=17, va="center", color=COLORS["ink"], alpha=0)
    bullet_dots.append(d); bullet_txts.append(t)
bullet_appear = [INTRO_END - 6, INTRO_END + 8, IND_END + 8, BATCH_END + 4]

# ---- Figure-level text (centered for vertical) ----
pill = fig.text(0.5, 0.978, "For Kids: Middle School", ha="center", va="center",
                fontsize=15, color="white",
                bbox=dict(boxstyle="round,pad=0.45", fc=COLORS["navy"], ec="none", alpha=0.85))
title_main = fig.text(0.5, 0.945, "The Law of Large Numbers", ha="center", fontsize=30,
                      fontweight="bold", color=COLORS["ink"], alpha=0)
title_sub = fig.text(0.5, 0.920, "Why many tries beat one lucky roll", ha="center",
                     fontsize=17, color=COLORS["grey"], alpha=0)
watermark = fig.text(0.035, 0.013, "\u00A9 Mugambi Ndwiga / @craftsandengineering",
                     fontsize=12, color=COLORS["navy"], alpha=0.6, ha="left")

# ---- Closing card ----
ax_close.add_patch(Rectangle((0, 0), 1, 1, fc=COLORS["navy"], ec="none"))
ax_close.text(0.5, 0.54, "Made by Mugambi Ndwiga", ha="center", va="center",
              fontsize=30, fontweight="bold", color="white")
ax_close.text(0.5, 0.47, "@craftsandengineering", ha="center", va="center",
              fontsize=20, color="white")

# --------------------------------------------------------------------------- #
# UPDATE
# --------------------------------------------------------------------------- #
def update(f):
    st = frame_state(f)
    closing = st["phase"] == "close"
    for ax in (ax_ramp, ax_conv, ax_panel):
        ax.set_visible(not closing)
    ax_close.set_visible(closing)
    if closing:
        for t in (title_main, title_sub, pill, watermark):
            t.set_alpha(0)
        return ()

    title_main.set_alpha(fade(f, 6, 24))
    title_sub.set_alpha(fade(f, 14, 24))
    pill.set_alpha(0.95)
    watermark.set_alpha(0.6)
    a = fade(f, 30, 20)
    apex_label.set_alpha(a); arr_l.set_alpha(a); arr_r.set_alpha(a)

    n = st["n"]
    L = int(lefts_cum[n]); R = n - L
    bar_left.set_height(bar_height(L)); bar_right.set_height(bar_height(R))
    count_left.set_text("$L_n$ = %d" % L)
    count_right.set_text("$R_n$ = %d" % R)
    counter_n.set_text("n = %d  balls" % n)
    counter_phat.set_text("" if n == 0 else "$\\hat{p}_n$ = %.3f" % proportion[n])

    for dot in flight_dots:
        dot.set_alpha(0)
    if st["phase"] == "intro":
        flight_dots[0].set_data([0.0], [DROP_TOP])           # ball poised at the gate
        flight_dots[0].set_color(COLORS["navy"])
        flight_dots[0].set_alpha(fade(f, 30, 16))
    else:
        stream_fade = fade(f, IND_END, 12) if st["phase"] == "many" else 1.0
        for k, fl in enumerate(st["flights"]):
            side, t, alpha = fl
            x, y = ball_position(side, t)
            dot = flight_dots[k]
            dot.set_data([x], [y])
            dot.set_color(COLORS["sage"] if side else COLORS["coral"])
            vanish = 1.0 if t < 0.85 else max(0.0, (1 - t) / 0.15)
            dot.set_alpha(alpha * vanish * stream_fade)

    if n >= 1:
        conv_line.set_data(trials[1:n + 1], proportion[1:n + 1])
        conv_head.set_data([n], [proportion[n]])
    else:
        conv_line.set_data([], []); conv_head.set_data([], [])

    for d, t, appear in zip(bullet_dots, bullet_txts, bullet_appear):
        av = fade(f, appear, 16)
        d.set_alpha(av); t.set_alpha(av)

    if st["phase"] == "reveal":
        pulse = 0.5 + 0.5 * np.sin((f - BATCH_END) * 0.22)
        conv_note.set_alpha(fade(f, BATCH_END + 6, 18))
        bullet_txts[-1].set_fontsize(17 + 2.0 * pulse)
    else:
        conv_note.set_alpha(0)
    return ()

# --------------------------------------------------------------------------- #
# RENDER
# --------------------------------------------------------------------------- #
anim = FuncAnimation(fig, update, frames=TOTAL, interval=1000 / C["fps"], blit=False)
writer = FFMpegWriter(fps=C["fps"], codec="libx264",
                      extra_args=["-pix_fmt", "yuv420p", "-preset", "medium", "-crf", "24"])
anim.save(C["outfile"], writer=writer, dpi=C["dpi"])
plt.close(fig)

import os
print("Saved %s  (%.2f MB, %.1f s, 1080x1920 vertical 9:16)" %
      (C["outfile"], os.path.getsize(C["outfile"]) / 1e6, TOTAL / C["fps"]))

# --------------------------------------------------------------------------- #
# DISPLAY + DOWNLOAD  (least problematic across Colab / Jupyter)
# --------------------------------------------------------------------------- #
from IPython.display import Video, FileLink, display

display(Video(C["outfile"], embed=True, width=360,
              html_attributes="controls autoplay loop muted playsinline"))

def download_video(path=C["outfile"]):
    """Save the video locally (Colab opens a download dialog; otherwise a link)."""
    try:
        from google.colab import files
        files.download(path)
    except Exception:
        display(FileLink(path))

download_video()

Saved law_of_large_numbers_vertical.mp4  (0.73 MB, 36.2 s, 1080x1920 vertical 9:16)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>